# Commodity futures: liquidity, releases, and curve risk

This notebook applies Screamer to three causal commodity-futures workflows:

- trade-tape liquidity from TickRuleSign, SignedVolume, and VPIN;
- a scheduled inventory surprise whose usable time is later than its release time;
- a one-to-one calendar-spread target with Yang-Zhang risk scaling and a costed OHLC backtest.

The inputs are seeded synthetic data, so the notebook runs offline and makes no claim about an investable signal. The examples show data availability, execution timing, and operator composition that transfer to a real futures feed.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from screamer import (
    BacktestOHLCTarget,
    CombineLatest,
    Delay,
    NegPart,
    PosPart,
    RollingYangZhangVol,
    RollingZscore,
    SignedVolume,
    Sub,
    TickRuleSign,
    VPIN,
    backtest_report,
)


## Trade-tape liquidity

Some commodity venues provide aggressor flags, while others provide only price and size. TickRuleSign infers the sign from successive prices. SignedVolume turns that sign and the traded contract count into signed flow. VPIN then measures buy-versus-sell imbalance on a volume clock, rather than on an arbitrary wall-clock bar.


In [ ]:
rng = np.random.default_rng(21)
n_trades = 720
minute = 60_000
trade_ts = np.arange(n_trades, dtype=np.int64) * minute

# A deterministic, irregular-looking futures tape with a short directional episode.
increments = 0.012 * rng.standard_normal(n_trades)
increments[300:380] += 0.018
trade_price = 74.0 + np.cumsum(increments)
trade_price = np.round(trade_price, 2)
contracts = rng.integers(1, 50, size=n_trades).astype(np.float64)

trade_sign = TickRuleSign()(trade_price)
signed_volume = SignedVolume()(trade_sign, contracts)
buy_volume = PosPart()(signed_volume)
sell_volume = NegPart()(signed_volume)

bucket_volume = contracts.sum() / 24.0
vpin = VPIN(bucket_volume=bucket_volume, n_buckets=8)(buy_volume, sell_volume)

fig, (ax_price, ax_vpin) = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
ax_price.plot(trade_ts / minute, trade_price, color="0.2", lw=0.9)
ax_price.set_ylabel("futures price")
ax_price.set_title("Synthetic futures tape and order-flow toxicity")
ax_vpin.plot(trade_ts / minute, vpin, color="crimson", lw=1.0)
ax_vpin.set_ylabel("VPIN")
ax_vpin.set_xlabel("minutes")
ax_vpin.set_ylim(0, 1)
fig.tight_layout()


The same operators process an event at a time. This assertion checks that the historical calculation and the live path have identical values.


In [ ]:
live_sign = TickRuleSign()
live_signed_volume = SignedVolume()
live_buy = PosPart()
live_sell = NegPart()
live_vpin = VPIN(bucket_volume=bucket_volume, n_buckets=8)
vpin_live = []

for price, size in zip(trade_price, contracts):
    sign = live_sign(price)
    flow = live_signed_volume(sign, size)
    vpin_live.append(live_vpin(live_buy(flow), live_sell(flow)))

np.testing.assert_allclose(vpin, np.asarray(vpin_live), equal_nan=True)
print("VPIN matches when the tape is processed one event at a time.")


## Release time is not availability time

A scheduled EIA, WASDE, or exchange-stock observation is not usable at its scheduled timestamp if parsing, validation, or dissemination takes time. Delay re-stamps the surprise at its availability time. CombineLatest performs the as-of join with the futures tape and waits until both inputs have a value.


In [ ]:
release_ts = np.array([180, 360, 540], dtype=np.int64) * minute
inventory_surprise = np.array([-0.8, 0.35, -0.2])
dissemination_delay = 15 * minute

available_surprise, available_ts = Delay(dissemination_delay)(inventory_surprise, release_ts)
joined, joined_ts = CombineLatest(emit="when_all")(
    (trade_price, trade_ts),
    (available_surprise, available_ts),
)

np.testing.assert_array_equal(available_ts, release_ts + dissemination_delay)
assert np.all(joined_ts >= available_ts[0])
np.testing.assert_allclose(joined[0, 1], inventory_surprise[0])

fig, ax = plt.subplots(figsize=(9, 2.8))
ax.step(joined_ts / minute, joined[:, 1], where="post", color="indigo")
ax.scatter(available_ts / minute, inventory_surprise, color="crimson", zorder=3, label="available surprise")
ax.set_xlabel("minutes")
ax.set_ylabel("inventory surprise")
ax.set_title("The release enters the joined stream only after its availability delay")
ax.legend(loc="upper right")
fig.tight_layout()


## Calendar-spread target and OHLC risk

The two contracts below use the same multiplier, so their one-to-one difference is a price spread. In production, normalize unequal multipliers and use a contract-roll policy before this step. An OHLC calendar spread must come from executable spread bars or timestamp-level leg matching. It cannot be assembled as the difference of independent leg highs and lows.


In [ ]:
n_days = 252
daily_ts = np.arange(n_days, dtype=np.int64) * 86_400_000

# Simulate a front contract and an explicitly tradable one-to-one calendar spread.
front_close = 75.0 * np.exp(np.cumsum(0.0002 + 0.009 * rng.standard_normal(n_days)))
spread_close = np.empty(n_days)
spread_close[0] = 0.35
for i in range(1, n_days):
    spread_close[i] = 0.92 * spread_close[i - 1] + 0.09 * rng.standard_normal()
next_close = front_close - spread_close

spread_open = np.r_[spread_close[0], spread_close[:-1]] + 0.025 * rng.standard_normal(n_days)
spread_high = np.maximum(spread_open, spread_close) + 0.04 + 0.03 * rng.random(n_days)
spread_low = np.minimum(spread_open, spread_close) - 0.04 - 0.03 * rng.random(n_days)

calendar_close = Sub()(front_close, next_close)
np.testing.assert_allclose(calendar_close, spread_close)

zscore = RollingZscore(window_size=20)(calendar_close)
yang_zhang = RollingYangZhangVol(window_size=20)(
    spread_open, spread_high, spread_low, calendar_close
)

risk_unit = np.nanmedian(yang_zhang[40:])
target = np.zeros(n_days)
ready = np.isfinite(zscore) & np.isfinite(yang_zhang) & (yang_zhang > 0)
target[ready] = np.clip(-zscore[ready] * risk_unit / yang_zhang[ready], -2.0, 2.0)

backtest = BacktestOHLCTarget(
    taker_fee=0.0002, tick_size=0.01, min_position=-2.0, max_position=2.0
)(target, spread_open, spread_high, spread_low, calendar_close)
summary = backtest_report(backtest)[1]

for name in ("total_pnl", "total_cost", "num_trades", "max_drawdown"):
    print(f"{name:14s} {summary[name]:10.4f}")


BacktestOHLCTarget receives the target computed from each spread close and executes it at the next spread open. No manual lag is added. The target below is a mechanical example, not a calibrated production rule.


In [ ]:
day = np.arange(n_days)
fig, axes = plt.subplots(4, 1, sharex=True, figsize=(9, 8))

axes[0].plot(day, calendar_close, color="0.2", lw=0.9)
axes[0].set_ylabel("spread")
axes[0].set_title("Calendar-spread target, risk estimate, and costed backtest")

axes[1].plot(day, zscore, color="indigo", lw=0.9, label="rolling z-score")
axes[1].plot(day, target, color="darkorange", lw=0.9, label="target")
axes[1].axhline(0, color="0.5", lw=0.6)
axes[1].set_ylabel("z / target")
axes[1].legend(loc="upper right", fontsize=8)

axes[2].plot(day, yang_zhang, color="seagreen", lw=0.9)
axes[2].set_ylabel("Yang-Zhang vol")

axes[3].plot(day, backtest[:, 0], color="steelblue", lw=0.9)
axes[3].axhline(0, color="0.5", lw=0.6)
axes[3].set_ylabel("equity")
axes[3].set_xlabel("trading day")
fig.tight_layout()


## Applying the pattern to production data

- Preserve the first timestamp at which each release was parsed and validated. Use that timestamp, not the scheduled release time, as the input to Delay.
- Keep trade-price, size, and contract fields at their native event resolution for liquidity measures. Do not infer intraday toxicity from a daily close.
- Maintain multiplier, currency, delivery month, and roll metadata outside the numerical stream. Feed Screamer normalized, economically comparable series.
- Record batch-versus-live equality tests when a research graph becomes a monitored live process.
